# Shohoj Lipi - Sentence Router Training
BanglaBERT fine-tuning on 96K Bengali sentences (Simple/Complex classification)

In [1]:
!pip install -q transformers datasets evaluate scikit-learn accelerate
print('Packages installed.')

Packages installed.


In [2]:
import os, urllib.request
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
import evaluate
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)

# ----- GPU INFO -----
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    cap  = torch.cuda.get_device_capability(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} | compute cap {cap} | {mem:.1f} GB')

2026-06-30 17:35:51.290711: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782840951.317377     263 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782840951.326107     263 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782840951.349296     263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782840951.349321     263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782840951.349323     263 computation_placer.cc:177] computation placer alr

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 | compute cap (7, 5) | 15.6 GB
  GPU 1: Tesla T4 | compute cap (7, 5) | 15.6 GB


In [3]:
# ----- DOWNLOAD DATA -----
os.makedirs('data', exist_ok=True)
base = 'https://raw.githubusercontent.com/tafseer-nayeem/BengaliReadability/main/Data/Learning/'
for fname in ['train.csv', 'valid.csv', 'test.csv']:
    dest = f'data/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(base + fname, dest)
    else:
        print(f'Already exists: {fname}')

df_train = pd.read_csv('data/train.csv')
df_valid = pd.read_csv('data/valid.csv')
print(f'Train rows: {len(df_train)} | Valid rows: {len(df_valid)}')
print(f'Columns: {df_train.columns.tolist()}')
print(f'Label distribution (train):\n{df_train.iloc[:, 1].value_counts()}')

Already exists: train.csv
Already exists: valid.csv
Already exists: test.csv
Train rows: 91935 | Valid rows: 2200
Columns: ['sentence', 'class']
Label distribution (train):
class
0    54033
1    37902
Name: count, dtype: int64


In [4]:
# ----- PREPARE LABELS -----
text_col  = df_train.columns[0]
label_col = df_train.columns[1]

if df_train[label_col].dtype == object:
    unique_labels = sorted(df_train[label_col].unique())
    label2id = {l: i for i, l in enumerate(unique_labels)}
    id2label  = {i: l for l, i in label2id.items()}
    df_train[label_col] = df_train[label_col].map(label2id)
    df_valid[label_col] = df_valid[label_col].map(label2id)
    print(f'Label map: {label2id}')
else:
    label2id = {0: 'Simple', 1: 'Complex'}
    id2label  = {0: 'Simple', 1: 'Complex'}
    print('Labels are already numeric.')

ds = DatasetDict({
    'train': Dataset.from_pandas(
        df_train[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}),
        preserve_index=False
    ),
    'validation': Dataset.from_pandas(
        df_valid[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}),
        preserve_index=False
    ),
})
print(f'Dataset: {ds}')

Labels are already numeric.
Dataset: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 91935
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2200
    })
})


In [5]:
# ----- TOKENIZE -----
MODEL_NAME = 'csebuetnlp/banglabert_small'
print(f'Loading tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded. Starting tokenization of dataset (this may take 2-4 mins)...')

def preprocess(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128, padding=False)

tokenized_ds = ds.map(preprocess, batched=True, batch_size=1000, desc='Tokenizing')
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f'Tokenization complete. Train size: {len(tokenized_ds["train"])} | Val size: {len(tokenized_ds["validation"])}')

Loading tokenizer: csebuetnlp/banglabert_small...
Tokenizer loaded. Starting tokenization of dataset (this may take 2-4 mins)...


Tokenizing:   0%|          | 0/91935 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2200 [00:00<?, ? examples/s]

Tokenization complete. Train size: 91935 | Val size: 2200


In [6]:
# ----- SANITY CHECK: verify GPU works with a tiny forward pass -----
print('Running GPU sanity check...')
test_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, torch_dtype=torch.float32
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_model = test_model.to(device)
dummy_ids  = torch.ones(2, 32, dtype=torch.long).to(device)
dummy_mask = torch.ones(2, 32, dtype=torch.long).to(device)
with torch.no_grad():
    out = test_model(input_ids=dummy_ids, attention_mask=dummy_mask)
print(f'Sanity check PASSED. Output logits shape: {out.logits.shape}')
del test_model
torch.cuda.empty_cache()
print('GPU memory cleared. Ready to train.')

`torch_dtype` is deprecated! Use `dtype` instead!


Running GPU sanity check...


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert_small and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sanity check PASSED. Output logits shape: torch.Size([2, 2])
GPU memory cleared. Ready to train.


In [7]:
# ----- LOAD MODEL -----
print(f'Loading model: {MODEL_NAME}...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float32,
)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded. Total params: {total_params:.1f}M')
print(f'Will use {torch.cuda.device_count()} GPU(s) for training.')

Loading model: csebuetnlp/banglabert_small...


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert_small and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded. Total params: 13.7M
Will use 2 GPU(s) for training.


In [8]:
# ----- TRAIN -----
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

# Use both GPUs: Trainer handles DataParallel automatically when CUDA_VISIBLE_DEVICES is not restricted
training_args = TrainingArguments(
    output_dir='./sentence_router_pt',
    learning_rate=2e-5,
    per_device_train_batch_size=32,   # 32 per GPU = 64 effective batch size on 2xT4
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=False,
    bf16=False,
    report_to='none',
    dataloader_num_workers=2,
    logging_steps=100,
    logging_strategy='steps',
    disable_tqdm=False,
    optim='adamw_torch',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Starting training...')
print(f'Steps per epoch: {len(tokenized_ds["train"]) // (32 * max(1, torch.cuda.device_count()))}')
trainer.train()
trainer.save_model('sentence_router_pt')
print('Training complete! Model saved to sentence_router_pt/')

/tmp/ipykernel_263/1471700982.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...
Steps per epoch: 1436


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.399100,0.371550,0.830909
2,0.351500,0.338472,0.848636
3,0.341400,0.332569,0.850909


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training complete! Model saved to sentence_router_pt/


In [9]:
# ----- EXPORT TO ONNX -----
print('Exporting to ONNX...')
!optimum-cli export onnx --model sentence_router_pt --task text-classification sentence_router_onnx
print('ONNX export done!')
print('Download the sentence_router_onnx/ folder from the Output tab and place it under models/ in your backend.')

Exporting to ONNX...
2026-06-30 17:42:27.831940: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782841347.855756    9118 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782841347.863853    9118 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782841347.885300    9118 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782841347.885332    9118 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782841347.885335    9118 computation_placer.cc:177] c